## 功能教程

https://blog.csdn.net/tianjuewudi/article/details/123113885

## 安装

**使用pip安装stable-baselines3有问题，请前往github下载最新的zip**

现在stable_baselines3的代码中已经使用gymnasium的格式：env = gym.make("CartPole-v1", render_mode="rgb_array")

如果是使用的是gym的格式，语法应该是：env.render(mode='human')

但是现在如果使用pip安装的旧包，在所有例子中都会提示asserterror action space的类型不支持

因此，应该前往[github](https://github.com/DLR-RM/stable-baselines3)下载最新的包

## 使用自定义网络训练 （GPT）

### 安装依赖库

确保你安装了以下Python库：

In [ ]:
pip install stable-baselines3 torch

### 构建自定义3D卷积网络

我们将创建一个基本的3D卷积网络类，该类继承自torch.nn.Module。

In [ ]:
import torch
import torch.nn as nn
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

class Custom3DCNN(BaseFeaturesExtractor):
    def __init__(self, observation_space, features_dim):
        super(Custom3DCNN, self).__init__(observation_space, features_dim)
        
        # 例如，输入是 (channels, depth, height, width)
        self.cnn = nn.Sequential(
            nn.Conv3d(in_channels=observation_space.shape[0], out_channels=16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool3d(kernel_size=2, stride=2),
            nn.Conv3d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool3d(kernel_size=2, stride=2),
            nn.Flatten()
        )
        
        # 用于确定线性层输入的大小
        with torch.no_grad():
            self.output_dim = self.cnn(torch.as_tensor(observation_space.sample()[None]).float()).shape[1]

        self.linear = nn.Linear(self.output_dim, features_dim)

    def forward(self, observations):
        return self.linear(self.cnn(observations))


### 创建PPO模型并训练

现在我们可以使用自定义的3D卷积网络来创建PPO模型。首先需要定义你的环境（环境需要支持3D观察空间），然后创建并训练PPO模型。

In [ ]:
from stable_baselines3 import PPO
from gym import spaces

# 假设观察空间是4个通道的64x64x64体数据
observation_space = spaces.Box(low=0, high=1, shape=(4, 64, 64, 64), dtype=np.float32)
features_dim = 256  # 特征维度

model = PPO("MlpPolicy", env, policy_kwargs=dict(
    features_extractor_class=Custom3DCNN,
    features_extractor_kwargs=dict(features_dim=features_dim),
    net_arch=[128, 128]  # 可以根据需要调整策略和值网络的体系结构
), verbose=1)

model.learn(total_timesteps=10000)

请注意，在实际应用中，你需要根据具体的环境来调整网络架构和参数。上面的示例仅提供了一个基本框架。你还需要有一个兼容的环境，其观察空间匹配网络输入的形状和类型。如果你有特定的环境或进一步的需求，可以进一步调整代码。

## 归一化 （GPT）


Stable Baselines3 是一个在 Python 中用于深度强化学习的流行库，它提供了多种算法的实现，如 A2C, PPO, SAC 等。在 Stable Baselines3 中，状态归一化通常是通过环境包装器来实现的。一个常用的包装器是 VecNormalize，它可以对环境的观测和回报进行归一化。

下面是如何使用 VecNormalize 包装器对状态进行归一化的示例代码：

In [ ]:
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3 import PPO
import gym

# 创建环境
env = gym.make('CartPole-v1')

# 将环境包装到 DummyVecEnv 中以允许向量化
env = DummyVecEnv([lambda: env])

# 使用 VecNormalize 对状态和奖励进行归一化
env = VecNormalize(env, norm_obs=True, norm_reward=False)

# 创建模型
model = PPO('MlpPolicy', env, verbose=1)

# 训练模型
model.learn(total_timesteps=10000)

# 保存归一化的环境
env.save("vec_normalize.pkl")

# 在之后加载归一化环境时
# 注意：加载环境时需要使用相同结构的 VecNormalize
loaded_env = VecNormalize.load("vec_normalize.pkl", env)

# 创建新模型以在归一化环境中使用
new_model = PPO('MlpPolicy', loaded_env, verbose=1)


## Getting started

In [2]:
import gymnasium as gym
from stable_baselines3 import A2C

env = gym.make("CartPole-v1", render_mode="rgb_array")

model = A2C("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=10_000)

vec_env = model.get_env()
obs = vec_env.reset()
for i in range(1000):
    action, _state = model.predict(obs, deterministic=True)
    obs, reward, done, info = vec_env.step(action)
    vec_env.render("human")
    # VecEnv resets automatically
    # if done:
    #   obs = vec_env.reset()

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 53.5     |
|    ep_rew_mean        | 53.5     |
| time/                 |          |
|    fps                | 174      |
|    iterations         | 100      |
|    time_elapsed       | 2        |
|    total_timesteps    | 500      |
| train/                |          |
|    entropy_loss       | -0.563   |
|    explained_variance | 0.0706   |
|    learning_rate      | 0.0007   |
|    n_updates          | 99       |
|    policy_loss        | 0.736    |
|    value_loss         | 7.73     |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 52.4     |
|    ep_rew_mean        | 52.4     |
| time/                 |          |
|    fps                | 242      |
|    iterations         | 200      |
|    time_elapsed

## TD3

In [1]:
import gymnasium as gym
import numpy as np

from stable_baselines3 import TD3
from stable_baselines3.common.noise import NormalActionNoise, OrnsteinUhlenbeckActionNoise

env = gym.make("Pendulum-v1", render_mode="rgb_array")

# The noise objects for TD3
n_actions = env.action_space.shape[-1]
action_noise = NormalActionNoise(mean=np.zeros(n_actions), sigma=0.1 * np.ones(n_actions))

model = TD3("MlpPolicy", env, action_noise=action_noise, verbose=1)
model.learn(total_timesteps=10000, log_interval=10)
model.save("td3_pendulum")
vec_env = model.get_env()

del model # remove to demonstrate saving and loading

model = TD3.load("td3_pendulum")

obs = vec_env.reset()
while True:
    action, _states = model.predict(obs)
    obs, rewards, dones, info = vec_env.step(action)
    vec_env.render("human")

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 200       |
|    ep_rew_mean     | -1.34e+03 |
| time/              |           |
|    episodes        | 10        |
|    fps             | 154       |
|    time_elapsed    | 12        |
|    total_timesteps | 2000      |
| train/             |           |
|    actor_loss      | 29.1      |
|    critic_loss     | 0.0844    |
|    learning_rate   | 0.001     |
|    n_updates       | 1800      |
----------------------------------
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 200       |
|    ep_rew_mean     | -1.29e+03 |
| time/              |           |
|    episodes        | 20        |
|    fps             | 157       |
|    time_elapsed    | 25        |
|    total_timesteps | 4000      |
| train/             |           |
|    actor_loss      | 56.7   

KeyboardInterrupt: 

## PPO

In [ ]:
import gymnasium as gym

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

# Parallel environments
vec_env = make_vec_env("CartPole-v1", n_envs=4)

model = PPO("MlpPolicy", vec_env, verbose=1)
model.learn(total_timesteps=25000)
model.save("ppo_cartpole")

del model # remove to demonstrate saving and loading

model = PPO.load("ppo_cartpole")

obs = vec_env.reset()
while True:
    action, _states = model.predict(obs)
    obs, rewards, dones, info = vec_env.step(action)
    vec_env.render("human")

In [2]:
help(PPO)

Help on class PPO in module stable_baselines3.ppo.ppo:

class PPO(stable_baselines3.common.on_policy_algorithm.OnPolicyAlgorithm)
 |  PPO(policy: Union[str, Type[stable_baselines3.common.policies.ActorCriticPolicy]], env: Union[gymnasium.core.Env, stable_baselines3.common.vec_env.base_vec_env.VecEnv, str], learning_rate: Union[float, Callable[[float], float]] = 0.0003, n_steps: int = 2048, batch_size: int = 64, n_epochs: int = 10, gamma: float = 0.99, gae_lambda: float = 0.95, clip_range: Union[float, Callable[[float], float]] = 0.2, clip_range_vf: Union[NoneType, float, Callable[[float], float]] = None, normalize_advantage: bool = True, ent_coef: float = 0.0, vf_coef: float = 0.5, max_grad_norm: float = 0.5, use_sde: bool = False, sde_sample_freq: int = -1, target_kl: Optional[float] = None, stats_window_size: int = 100, tensorboard_log: Optional[str] = None, policy_kwargs: Optional[Dict[str, Any]] = None, verbose: int = 0, seed: Optional[int] = None, device: Union[torch.device, str] 

## logger

https://blog.csdn.net/qq_47997583/article/details/123670109

In [4]:
import stable_baselines3 as sb
from stable_baselines3 import PPO
from stable_baselines3.common.logger import configure

path = r'E:\\Tem'

logger = configure(path, ['stdout', 'log', 'json'])
model = PPO('MlpPolicy', 'CartPole-v1', verbose=2, n_steps=1024)
model.set_logger(logger)

model.learn(10000)

Logging to E:\\Tem
Using cuda device
Creating environment from the given name 'CartPole-v1'
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 22.3     |
|    ep_rew_mean     | 22.3     |
| time/              |          |
|    fps             | 556      |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 1024     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 24.5        |
|    ep_rew_mean          | 24.5        |
| time/                   |             |
|    fps                  | 448         |
|    iterations           | 2           |
|    time_elapsed         | 4           |
|    total_timesteps      | 2048        |
| train/                  |             |
|    approx_kl            | 0.007920092 |
|    clip_fraction        | 0

## Monitor

https://zhuanlan.zhihu.com/p/657935696

## Custom policy

In [ ]:
import torch as th
import torch.nn as nn
from gymnasium import spaces

from stable_baselines3 import PPO
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor


class CustomCNN(BaseFeaturesExtractor):
    """
    :param observation_space: (gym.Space)
    :param features_dim: (int) Number of features extracted.
        This corresponds to the number of unit for the last layer.
    """

    def __init__(self, observation_space: spaces.Box, features_dim: int = 256):
        super().__init__(observation_space, features_dim)
        # We assume CxHxW images (channels first)
        # Re-ordering will be done by pre-preprocessing or wrapper
        n_input_channels = observation_space.shape[0]
        self.cnn = nn.Sequential(
            nn.Conv2d(n_input_channels, 32, kernel_size=8, stride=4, padding=0),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=0),
            nn.ReLU(),
            nn.Flatten(),
        )

        # Compute shape by doing one forward pass
        with th.no_grad():
            n_flatten = self.cnn(
                th.as_tensor(observation_space.sample()[None]).float()
            ).shape[1]

        self.linear = nn.Sequential(nn.Linear(n_flatten, features_dim), nn.ReLU())

    def forward(self, observations: th.Tensor) -> th.Tensor:
        return self.linear(self.cnn(observations))

policy_kwargs = dict(
    features_extractor_class=CustomCNN,
    features_extractor_kwargs=dict(features_dim=128),
)
model = PPO("CnnPolicy", "BreakoutNoFrameskip-v4", policy_kwargs=policy_kwargs, verbose=1)
model.learn(1000)